# 第14课：LangGraph 状态图与 Agent 骨架

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter14_LangGraph状态图_课后练习.ipynb](chapter14_LangGraph状态图_课后练习.ipynb)。

**阶段定位**：阶段一 · 前置理论。前置理论，后续 OpenClaw 直接复用状态图

第 13 课解决“模型在哪里、接口长什么样”。本课解决“Agent 怎么想下一步”。答案不是再写一个更长的 if-else，而是**状态图**。

## 学习目标

1. 用 Pydantic 或 TypedDict 思路声明全局状态，并指出哪些字段用累积 reducer、哪些用覆盖。
2. 用 StateGraph 连接 Agent 节点与 Tool 节点，形成闭环。
3. 写出条件边：根据 `need_tool` 决定去 tool 还是 finalize。
4. 设置 recursion_limit，并用 MemorySaver 做同 thread 的多轮恢复。

## 学习知识点

| 状态 | 拓扑 | 熔断与记忆 |
| --- | --- | --- |
| channels 声明字段 | add_node | recursion_limit |
| add_messages 累积 | add_edge | MemorySaver |
| overwrite 覆盖 | add_conditional_edges | thread_id |

## 基础回顾与案例提问

1. **R.1** 若把用户本轮输入和历史消息放进同一个列表并每次覆盖整个列表，多轮对话会丢什么？
2. **R.2** 条件边的 router 返回 `"tool"`，但 mapping 只有 `{"end": "finalize"}`，运行时会发生什么？
3. **R.3** 节点自己指向自己且 recursion_limit=3，应看到什么异常？

教学图在 `agent_lab.graph`。真实 LangGraph 的 API 同名概念可对照，本课不要求 pip install langgraph。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. Agent 本质是状态图

### 理论知识

**节点做一件事，边决定下一件事。** START 进入 agent；agent 可去 tool 或 finalize；finalize 到 END。这就是最小 Agent。

### 案例：三个名字先记熟


In [ ]:
print("START -> agent")
print("agent --(need_tool)--> tool -> finalize -> END")
print("agent --(else)--> finalize -> END")


### 讲解

不要把“会说话的模型”等同于 Agent。没有状态和边，只是单次补全。

### 易错点与练习

1. **K1.1** Tool 节点如果直接回到自己而不经过 agent，会缺什么决策？
2. **K1.2** END 是节点函数还是图的终止标记？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 状态与 Reducer

### 理论知识

**Reducer 回答：新旧值如何合成。** `messages` 用 `add_messages` 做列表拼接；`need_tool`、`result` 用覆盖。

### 案例：看两种合并


In [ ]:
from agent_lab.graph import add_messages
print(add_messages(["a"], ["b"]))
print(add_messages(None, ["first"]))


### 讲解

第 14 课验收会打印次轮 `messages`：里面应能看到首轮 think/tool/final，这就是 reducer 的证据。

### 易错点与练习

1. **K2.1** user 本轮文本应该累积还是覆盖？为什么？
2. **K2.2** trace 若用覆盖，调试时会丢掉什么？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. Node：只返回增量

### 理论知识

**节点函数接收当前状态，返回要合并的增量字典。** 不要在节点里偷偷改传入的 dict 又同时 return，教学里统一 return 更新。

### 案例：一个只思考的 agent 节点


In [ ]:
def agent(state):
    text = state.get("user", "")
    need = "库存" in text
    return {"messages": ["think:" + text], "need_tool": need}
print(agent({"user": "查询库存"}))


### 讲解

节点保持短小：判断意图、调用工具、收束回复，分成三个节点，条件边才画得清。

### 易错点与练习

1. **K3.1** 若 agent 直接在函数里 print 库存数字而不更新 state，finalize 读得到吗？
2. **K3.2** `need_tool` 为什么不要放进 messages 字符串里用 in 去猜？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 普通边与条件边

### 理论知识

**普通边是固定下一站。条件边先跑 router，再用 mapping 翻译成节点名。**

### 案例：router 必须返回 mapping 的键


In [ ]:
def route(state):
    return "tool" if state.get("need_tool") else "end"
print(route({"need_tool": True}), route({"need_tool": False}))


### 讲解

标签 end 不等于 END 常量，它只是 mapping 里的钥匙，通常映射到 finalize。

### 易错点与练习

1. **K4.1** mapping 写成 `{"tool": "tool"}` 却漏了 `end`，用户打招呼时会怎样？
2. **K4.2** START 到 agent 应该用普通边还是条件边？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 搭一条最小闭环

### 理论知识

**agent →（条件）tool/finalize，tool → finalize，finalize → END。**

### 案例：编译并跑一轮


In [ ]:
from agent_lab.graph import END, START, StateGraph, add_messages

channels = {"messages": add_messages, "trace": add_messages, "need_tool": lambda o, n: n, "result": lambda o, n: n}

def agent(state):
    text = state.get("user", "")
    return {"messages": ["think:" + text], "need_tool": "库存" in text}

def tool_node(state):
    return {"messages": ["tool:stock=12"], "result": "WIDGET-X=12"}

def finalize(state):
    return {"messages": ["final:" + str(state.get("result") or "no-tool")]}

g = StateGraph(channels)
g.add_node("agent", agent).add_node("tool", tool_node).add_node("finalize", finalize)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", lambda s: "tool" if s.get("need_tool") else "end", {"tool": "tool", "end": "finalize"})
g.add_edge("tool", "finalize")
g.add_edge("finalize", END)
app = g.compile(recursion_limit=8)
out = app.invoke({"user": "查询 WIDGET-X 库存", "messages": []})
print([t["node"] for t in out["trace"]])


### 讲解

首轮轨迹应为 agent、tool、finalize。查询词不含“库存”时不应进 tool。

### 易错点与练习

1. **K5.1** 把 user 改成“你好”再跑，轨迹应少哪个节点？
2. **K5.2** tool 为什么不要直接 add_edge 回 agent 造成隐式死循环？本课先经过 finalize。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. Checkpointer 与多轮

### 理论知识

**MemorySaver 按 thread_id 存整份状态。** 同一 `u1` 的第二轮会先 merge 上一轮，再跑新的 user。

### 案例：同一 thread 再 invoke 一次


In [ ]:
from agent_lab.graph import MemorySaver
saver = MemorySaver()
app2 = g.compile(checkpointer=saver, recursion_limit=8)
cfg = {"configurable": {"thread_id": "u1"}}
app2.invoke({"user": "查询 WIDGET-X 库存", "messages": []}, cfg)
r2 = app2.invoke({"user": "只是打个招呼", "messages": []}, cfg)
print(r2["messages"])


### 讲解

次轮轨迹列表可能仍看得到首轮节点名，因为教学实现里 `trace` 也用 add_messages 累积。这不是 bug，是 reducer 演示。

### 易错点与练习

1. **K6.1** 换一个 thread_id 再打招呼，messages 里还应有库存结果吗？
2. **K6.2** Checkpointer 若只存最后一条 message，客服场景会丢什么？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. recursion_limit 熔断

### 理论知识

**图可以合法循环，但不能无限循环。** 自环边 + 很小的 limit 用来上课演示熔断。

### 案例：故意造一个炸环


In [ ]:
from agent_lab.graph import StateGraph, START, add_messages
boom = StateGraph({"messages": add_messages})
boom.add_node("loop", lambda s: {"messages": ["x"]})
boom.add_edge(START, "loop")
boom.add_edge("loop", "loop")
try:
    boom.compile(recursion_limit=3).invoke({"messages": []})
except RuntimeError as exc:
    print(exc)


### 讲解

真实项目里熔断后应回退到安全回复，而不是让 HTTP 500 漏到用户。本课先要求你能读懂异常字样。

### 易错点与练习

1. **K7.1** limit=3 时节点大约执行几次？
2. **K7.2** 把 limit 改成 8 却仍自环，验收脚本为什么仍应失败？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 对照第 15 课

### 理论知识

**OpenClaw 不会另造一套状态哲学。** 它把 session_id 映射为 thread_id，把探活接到同一张图的入口。

### 案例：记一句话


In [ ]:
print("LangGraph 状态 -> OpenClawConfig.session_id / timeout / model")


### 讲解

本课骨架不会说话也可以验收；下一课才把 Fake 聊天接进 agent 节点。

### 易错点与练习

1. **K8.1** 哪些字段你会放进 OpenClawConfig 而不是写进节点闭包？
2. **K8.2** 为什么 recursion_limit 要出现在 compile 而不是每个节点里？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：构图、多轮、熔断

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　画出闭环

按案例搭 agent/tool/finalize，invoke 一次库存查询，打印节点轨迹。


In [ ]:
# P1.1: build graph and print first-turn nodes.


### P1.2　同 thread 第二轮

对同一 thread_id 再 invoke 一句不含库存的话，打印累积 messages。


In [ ]:
# P1.2: second turn on the same thread_id.


### P1.3　触发熔断

自环图 + recursion_limit=3，捕获 RuntimeError 并打印。


In [ ]:
# P1.3: trigger recursion_limit.


课后请打开 [chapter14_LangGraph状态图_课后练习.ipynb](chapter14_LangGraph状态图_课后练习.ipynb)。P1 自己解释 reducer，P2 画错一条边并修正，P3 选做隔离两个 thread。
